In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent / 'src'))

In [ ]:
from abc import ABC, abstractmethod
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)

In [ ]:
from option import CallOption, PutOption
from underlying import GeometricBrownianMotion

In [ ]:
class MarketState:
    
    def __init__(self, runs, steps, maturity):
        self.runs = runs
        self.steps = steps
        length = steps + 1
        shape = (runs, length)

        self.time = np.linspace(0, maturity, length)

        self.cash = np.zeros(shape)
        #self.portfolio = np.zeros(shape)
        #self.int_delta_dS = np.zeros(shape)
        self.underlying = np.zeros(shape)
        self.option = np.zeros(shape)
        self.true_delta = np.zeros(shape)
        self.approx_delta = np.zeros(shape)
        self.gamma = np.zeros(shape)

        self.pnl = np.zeros(runs)

In [ ]:
class DeltaHedgingEngine:

    def __init__(self, option, model, steps_threshold=10000):
        self.option = option
        self.model = model
        self.s_threshold = steps_threshold
        self.state:MarketState = None

    def run(self, rate, sigma, runs, mesh, random_seed=None):
        if random_seed is not None:
            np.random.seed(random_seed)

        maturity = self.option.T
        temp = int(round(maturity * (1/mesh)))
        mesh_multiplier = int(self.s_threshold // temp) + 1
        steps = mesh_multiplier * temp
        length = steps + 1
        dt = maturity / steps

        data = MarketState(runs, steps, maturity)
        data.underlying = self.model.simulate(maturity, runs, dt, random_seed)
        data.option = self.option.price(data.underlying, data.time, rate, sigma)
        data.true_delta = self.option.delta(data.underlying, data.time, rate, sigma)
        data.gamma = self.option.gamma(data.underlying, data.time, rate, sigma)

        old_delta = data.true_delta[:, 0]
        data.approx_delta[:, 0] = old_delta
        data.cash[:, 0] = -data.option[:, 0] + data.true_delta[:, 0]*data.underlying[:, 0]
        #data.portfolio[:, 0] = data.option[:, 0]

        for j in range(1, length):

            data.cash[:, j] = np.exp(rate*dt)*data.cash[:, j-1]
            #data.int_delta_dS[:, j] = data.int_delta_dS[:, j-1] + old_delta*(data.underlying[:, j] - data.underlying[:, j-1])
            #data.portfolio[:, j] = data.option[:, j] - data.int_delta_dS[:, j]

            if j % mesh_multiplier:
                data.approx_delta[:, j] = old_delta
            else:
                data.approx_delta[:, j] = data.true_delta[:, j]
                data.cash[:, j] += (data.true_delta[:, j] - old_delta)*data.underlying[:, j]
                old_delta = data.true_delta[:, j]
        
        #data.int_delta_dS[:, -1] = data.int_delta_dS[:, -2] + old_delta*(data.underlying[:, -1] - data.underlying[:, -2])
        #data.portfolio[:, -1] = data.option[:, -1] - data.int_delta_dS[:, -1]
        
        data.cash[:, -1] = np.exp(rate*dt)*data.cash[:, -2]
        data.pnl[:] = data.option[:, -1] - old_delta*data.underlying[:, -1] + data.cash[:, -1]
        self.state = data

In [ ]:
spot = 100
drift = 0.08
maturity = 1
sigma = 0.2
mesh = 1/365
runs = 1000
override_threshold = 1000

underlying = GeometricBrownianMotion(spot, drift, sigma)
call = CallOption(100, maturity)

eng = DeltaHedgingEngine(call, underlying, override_threshold)
eng.run(0.03, sigma, runs, mesh)
mkt = eng.state

print(np.mean(mkt.pnl))
print(np.std(mkt.pnl))


In [ ]:
print(np.mean(mkt.pnl))
print(np.std(mkt.pnl))

In [ ]:
path_index = 1 #[0,1,2,3,4]

plt.figure(figsize=(8,5))

#plt.plot(mkt.time, mkt.underlying[path_index], label=f'Stock')
#plt.plot(mkt.time, mkt.option[path_index], label=f'Option')
plt.plot(mkt.time, mkt.true_delta[path_index], label=f'Delta')
plt.plot(mkt.time, mkt.approx_delta[path_index], label=f'Approx Delta')
#plt.plot(mkt.time, mkt.gamma[path_index], label=f'Gamma')
#plt.plot(mkt.time, mkt.cash[path_index], label=f'cash')
#plt.plot(mkt.time, mkt.portfolio[path_index], label=f'portfolio')
#plt.plot(mkt.time, mkt.int_delta_dS[path_index], label=f'int Delta dS')


plt.xlabel('Years')
plt.ylabel('Price')
plt.legend()
plt.grid(True)

In [ ]:
plt.hist(mkt.pnl, bins=100, density=True)

plt.xlabel("Value")
plt.ylabel("Frequency")
plt.title("Histogram of NumPy Array")